In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import pandas as pd
# uncomment the following line if using colab
# %pip install lmfit
import lmfit

Physical constants assuming electron in the box

In [ ]:
h = 6.626E-34 # kgm^2/s  (Planck's Constant)
m = 9.11E-31 # kg  (mass of an electron)
c = 3.00E8 # m/s (speed of light)

For each dye molecule, the required values are $\lambda_\text{max}$ in nm and the number of π-electrons in the conjugated chain. For testing, you can use the sample data for the 4,4'-cyanine series (Moog, 1991) provided in the code cell.

The output includes the length of the box, $L$, corresponding to that wavelength based on the equation

$$\lambda_\text{max} = {{8mcL^2}\over{h(N+1)}}$$

where $N$ is the number of π-electrons in the chain. Note that the various analyses may use number of electrons, number of electron pairs, number of bonds, or number of carbon atoms in the chain, but all of those are simply related to each other as shown in the code cell.


In [ ]:
# Replace the following arrays with student data
wavelengths = np.array([592, 708, 813, 929])
electrons = np.array([10, 12, 14, 16])

# Uncomment the following lines to allow student input of data
# nmolecules = input("Enter the number of molecules: ")
# wavelengthlist = []
# electronlist = []
# for i in range(int(nmolecules)):
#     wavelengthlist.append(float(input("Enter the wavelength (nm) of molecule " + str(i+1) + ": ")))
#     electronlist.append(float(input("Enter the number of π-electrons in molecule " + str(i+1) + ": ")))
# wavelengths = np.array(wavelengthlist)
# electrons = np.array(electronlist)

wavelength_meters = wavelengths * 1.e-9
bonds = electrons - 2
carbons = electrons - 3
electronpairs = electrons / 2
boxlength = np.sqrt(h * wavelength_meters * (electrons + 1) /(8 * m * c))
DE = h * c / wavelength_meters
df = pd.DataFrame({'wavelength (nm)': wavelengths, 'bonds': bonds, 'electrons': electrons, 'carbons': carbons, 'electron pairs': electronpairs, 'box length (m)': boxlength,
                   '∆E (J)': DE})
pd.set_option('display.precision', 2)
display(df)


The following cells calculate the fit from Moog, *J. Chem. Educ.*, 1991. The two parameter fit is for the average bondlength in the conjugated chain and a paramter, $\gamma$, that represents the additional length due to the polarizability at the ends of the chain.

$$L = (b \times l) + \gamma$$

where $b$ and $L$ are the number of bonds calculated box length for each dye, $l$ is the average bondlength, and $\gamma$ is the additional length.

### Moog fit using polyfit

polyfit is a numpy function that fits data to a polynomial. We'll use four arguments here:

* independent variable
* dependent variable
* polynomial degree
* `cov=True`, which returns a covariance matrix from which uncertainties can be calculated

polyfit then returns an array. Index 0 is an array containing the slope and intercept; index 1 is a 2x2 matrix with the squared uncertainties as the diagonal elements.


In [ ]:
# For the Moog fit, the number of bonds is the independent variable
# and the box length is the dependent variable.
fitMoogresults = np.polyfit(bonds, boxlength, 1, cov=True)

# Extract the slope (b) and intercept (γ) from the fit results, along with their uncertainties.
print("The average bondlength is", f"{fitMoogresults[0][0]:.2e} ± {np.sqrt(fitMoogresults[1][0][0]):.2e} m")
print("The value of γ is", f"{fitMoogresults[0][1]:.2e} ± {np.sqrt(fitMoogresults[1][1][1]):.2e} m")

# corrcoef is a separate function that can be used to calculate the R^2 value
print("R^2: ", f"{np.corrcoef(bonds, boxlength)[0, 1]**2:.2f}")

In [ ]:
# check fit with plot

bondlength = fitMoogresults[0][0]
gamma = fitMoogresults[0][1]
Lfit = bondlength * bonds + gamma

plt.plot(bonds, boxlength, marker='o', linestyle='', color='b')    # original data
plt.plot(bonds, Lfit,  marker='', linestyle='-', color='g')                # fit results
plt.xlabel("# of bonds")
plt.ylabel("L (m)")
plt.legend(['experiment', 'fit'], loc='lower right')

### Moog fit using lmfit

The lmfit package provides fitting to non-linear (and linear) functions. The approach is minimization of the sum of the squared residuals between the data and the fit.

We'll first work through the method using the Moog linear fit as an example (which we hope will give the same result as what we've already done).

First, we create a `Parameters` object, which is a dictionary of parameters. Each parameter has a name (or key) and an initial value. For the Moog fit, the two parameters are the bondlength and the additional length, $\gamma$.

In [ ]:
params_Moog = lmfit.Parameters()
params_Moog.add('bondlength', value = 1.39e-10)
params_Moog.add('gamma', value = 0.5)

We then declare a function that returns an array of residuals for our data based on the Moog model. The function takes the following three arguments: The `Parameters` object, the independent variable data (number of bonds) and the dependent variable data (box length).

In [ ]:
def fit_Moog(params, bonds, boxlength):
    # calculate the boxlength the model predicts for the current parameter values
    Lcalc = params['bondlength'] * bonds + params['gamma'] 
    # calculate the residuals (difference between model and data)
    residuals = boxlength - Lcalc
    return residuals

Now we're ready to invoke the minimize function of lmfit. For that, we need three arguments:

* The function that returns the residuals for our model
* The `Parameters` object
* An array that contains the independent and dependent variable data.

lmfit.minimize returns a complex object from which we can extract the optimized values of the parameters (through the `params` attributes) as well as the uncertainties from the covariance matrix (through the `covar` attribute). See the code cell for the syntax.

In [ ]:
resultMoog = lmfit.minimize(fit_Moog, params_Moog, args=(bonds, boxlength))

# Show everything that the fit provides
print(lmfit.fit_report(resultMoog))
print()

# Extract the parameter values and uncertainties
print("The average bondlength is", f"{resultMoog.params.valuesdict()['bondlength']:.2e} ± {np.sqrt(resultMoog.covar[0, 0]):.2e}", "m")
print("The value of γ is", f"{resultMoog.params.valuesdict()['gamma']:.2e} ± {np.sqrt(resultMoog.covar[1, 1]):.2e}", "m")

In [ ]:
# check fit with plot

bondlength = resultMoog.params.valuesdict()['bondlength']
gamma = resultMoog.params.valuesdict()['gamma']
Lfit = bondlength * bonds + gamma
# plot fit

plt.plot(bonds, boxlength, marker='o', linestyle='', color='b')    # original data
plt.plot(bonds, Lfit,  marker='', linestyle='-', color='g')                # fit results
plt.xlabel("# of bonds")
plt.ylabel("L (m)")
plt.legend(['experiment', 'fit'], loc='lower right')

### Shoemaker-Garland-Kuhn fit using lmfit

Shoemaker and Garland propose a fit based on Kuhn's original work on these conjugated dye systems. That fit also has two parameters: the bondlength $l$, and a parameter $\alpha$ that represents the fractional number of additional heavy atoms in the conjugated chain to represent the polarizability. The equation for the Shoemaker-Garland fit is

$$\lambda_\text{max} = {{8mcl^2}\over{h}} {{(p + 3 +\alpha)^2}\over{p+4}}$$

where $p$ is the number of carbon atoms in the conjugated chain.

In the following cells, we'll apply lmfit to find the optimized values of $l$ and $\alpha$.

In [ ]:
# Create the Parameters object with the two parameters. Shoemaker and Garland suggest that
# the value of alpha should be in the range [0.5, 1].

params_Kuhn = lmfit.Parameters()
params_Kuhn.add('bondlength', value = 1.39e-10)
params_Kuhn.add('alpha', value = 1)

In [ ]:
# Define the function that returns the residuals for the Shoemaker-Garland model.

def fit_Kuhn(params, p, lambdaobs):
    lambdacalc = 8 * m * c * params['bondlength']**2 * (p + 3 + params['alpha'])**2 / (h * (p + 4))
    resid = lambdaobs - lambdacalc
    return resid

In [ ]:
# Apply lmfit.minimize to optimize the parameter values

resultKuhn = lmfit.minimize(fit_Kuhn, params_Kuhn, args=(carbons, wavelength_meters))

print(lmfit.fit_report(resultKuhn))
print()
print("The average bondlength is", f"{resultKuhn.params.valuesdict()['bondlength']:.2e} ± {np.sqrt(resultKuhn.covar[0, 0]):.2e}", "m")
print("The value of α is", f"{resultKuhn.params.valuesdict()['alpha']:.2f} ± {np.sqrt(resultKuhn.covar[1, 1]):.2f}")

In [ ]:
# check fit with plot

bonds = resultKuhn.params.valuesdict()['bondlength']
alpha = resultKuhn.params.valuesdict()['alpha']
lambdaFit = 8 * m * c * bonds**2 * (carbons + 3 + alpha)**2 / (h * (carbons + 4))
# plot fit

plt.plot(carbons, wavelength_meters, marker='o', linestyle='', color='b')    # original data
plt.plot(carbons, lambdaFit,  marker='', linestyle='-', color='g')      # fit results
plt.xlabel("# of carbons")
plt.ylabel("λ (m)")
plt.legend(['experiment', 'fit'], loc='lower right')

Let's try the same approach using the Halpern-McBane approach. That turns out to incorporate elements of the Moog and Shoemaker-Garland-Kuhn models. They start with a Moog approach to the bondlength.

$$L=bl_b + l_\text{ext}$$

and then plug that into the formula for $\lambda_\text{max}$ as a function of $L$. But first, they convert to the number of electron pairs $p_e$ using $b=2(p_e-1)$. The result is

$$\lambda_\text{max} = {{8mc\left((2(p_e-1)l_b+l_\text{ext})\right)^2}\over{h(2p_e+1)}}$$

In [ ]:
# Define function for Halpern-McBane residuals

def fit_Halpern(params, p_e, lambdaobs):
    lambdacalc = 8 * m * c * (2 * (p_e - 1) * params['bondlength'] + params['l_ext'])**2 / (h * (2 * p_e + 1))
    resid = lambdaobs - lambdacalc
    return resid

In [ ]:
# Define params and perform fit

params_Halpern = lmfit.Parameters()
params_Halpern.add('bondlength', value = 1.39e-10)
params_Halpern.add('l_ext', value = 0.5e-10)

resultHalpern = lmfit.minimize(fit_Halpern, params_Halpern, args=(electronpairs, wavelength_meters))
print(lmfit.fit_report(resultHalpern))
print()
print("The average bondlength is", f"{resultHalpern.params.valuesdict()['bondlength']:.2e} ± {np.sqrt(resultHalpern.covar[0, 0]):.2e}", "m")
print("The value of l_ext is", f"{resultHalpern.params.valuesdict()['l_ext']:.2e} ± {np.sqrt(resultHalpern.covar[1, 1]):.2e}", "m")


In [ ]:
# check fit with plot
bondlength = resultHalpern.params.valuesdict()['bondlength']
l_ext = resultHalpern.params.valuesdict()['l_ext']
lambdaFit = 8 * m * c * (2 * (electronpairs - 1) * bondlength + l_ext)**2 / (h * (2 * electronpairs + 1))
# plot fit  
plt.plot(electronpairs, wavelength_meters, marker='o', linestyle='', color='b')    # original data
plt.plot(electronpairs, lambdaFit,  marker='', linestyle='-', color='g')      # fit results
plt.xlabel("# of electron pairs")
plt.ylabel("λ (m)")
plt.legend(['experiment', 'fit'], loc='lower right')
plt.show()